In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

100%|██████████| 5.76M/5.76M [00:00<00:00, 8.14MB/s]

Extracting files...


'dataset'

In [ ]:
!head dataset/train_small.csv

userId,movieId,rating,date
64777,16043,5.0,1997-07-01
64777,5112,2.0,1997-07-02
64777,24688,4.0,1997-07-02
64777,16639,4.0,1997-07-01
64777,13084,4.0,1997-07-02
64777,15142,4.0,1997-07-03
64777,5741,3.0,1997-07-03
64777,25503,3.0,1997-07-03
64777,2780,4.0,1997-07-01


baseline.py

In [ ]:
import csv

# 1. train_small.csv에서 전체 평균 계산
total, count = 0.0, 0
with open('dataset/train_small.csv', newline='') as f:
    for row in csv.DictReader(f):
        total += float(row['rating'])
        count += 1

mean_rating = total / count

# 2. test_small.csv를 읽어 모든 예측값을 평균으로 채워 저장
with open('dataset/test_small.csv', newline='') as fin, \
     open('baseline_solution.csv', 'w', newline='') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=['id', 'rating'])
    writer.writeheader()
    for row in reader:
        writer.writerow({'id': f"{row['userId']}_{row['movieId']}", 'rating': mean_rating})


In [ ]:
!head baseline_solution.csv

id,rating
64777_12355,3.572934302949529
64777_1864,3.572934302949529
64777_7104,3.572934302949529
64777_22572,3.572934302949529
64777_6751,3.572934302949529
42001_994,3.572934302949529
42001_11270,3.572934302949529
42001_23855,3.572934302949529
42001_19015,3.572934302949529


##Bias 모델 - 데이터 준비

### - userId와 itemId를 리매핑
sparse -> dense

### - users, items, ratings를 numpy array 형태로 변환

In [ ]:
import numpy as np

userIds, itemIds = {}, {}
n_users, n_items, n_ratings = 0,0,0
users, items, ratings = [], [], []

with open('dataset/train_small.csv') as f:
  for row in csv.DictReader(f):
    if row['userId'] not in userIds:
      userIds[row['userId']] = n_users
      n_users += 1
    if row['movieId'] not in itemIds:
      itemIds[row['movieId']] = n_items
      n_items += 1

    n_ratings += 1

    uid = userIds[row['userId']]
    iid = itemIds[row['movieId']]
    rating = float(row['rating'])

    users.append(uid)
    items.append(iid)
    ratings.append(rating)

  users = np.array(users)
  items = np.array(items)
  ratings = np.array(ratings)


##Bias 모델 - Gradient Descent

In [ ]:
import numpy as np

mean = ratings.mean()
user_bias = np.zeros(n_users)
item_bias = np.zeros(n_items)

lr = 0.0001
lmd = 0.001

for epoch in range(1000):
  h = mean + user_bias[users] + item_bias[items]
  diff = h - ratings
  print(f"epoch: {epoch}, mse: {(diff**2).mean()}")

  user_grad = np.bincount(users, weights=diff) + lmd*user_bias
  item_grad = np.bincount(items, weights=diff) + lmd*item_bias

  user_bias -= lr*user_grad
  item_bias -= lr*item_grad

/tmp/ipykernel_651/171413689.py:11: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  h = mean + user_bias[users] + item_bias[items]


epoch: 0, mse: 1.1046917649015608
epoch: 1, mse: 1.0579511051977446
epoch: 2, mse: 1.0263763732417537
epoch: 3, mse: 1.0022803025720906
epoch: 4, mse: 0.9826676009444287
epoch: 5, mse: 0.9661141221448383
epoch: 6, mse: 0.9518189891056277
epoch: 7, mse: 0.9392748170998848
epoch: 8, mse: 0.9281336155931886
epoch: 9, mse: 0.9181435338678295
epoch: 10, mse: 0.9091151766412008
epoch: 11, mse: 0.9009019538125752
epoch: 12, mse: 0.8933878171383889
epoch: 13, mse: 0.8864792060312128
epoch: 14, mse: 0.8800995421730495
epoch: 15, mse: 0.8741853430867348
epoch: 16, mse: 0.8686834044861761
epoch: 17, mse: 0.8635487110712154
epoch: 18, mse: 0.8587428573515882
epoch: 19, mse: 0.8542328339151203
epoch: 20, mse: 0.8499900808548053
epoch: 21, mse: 0.8459897399837275
epoch: 22, mse: 0.842210057300404
epoch: 23, mse: 0.8386319006183183
epoch: 24, mse: 0.8352383665814789
epoch: 25, mse: 0.8320144578464456
epoch: 26, mse: 0.8289468159089907
epoch: 27, mse: 0.8260234984692228
epoch: 28, mse: 0.8232337927468

In [ ]:
with open('dataset/test_small.csv') as fin, \
     open('bias_solution.csv','w', newline='') as fout:

    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=['id','rating'])
    writer.writeheader()

    for row in reader:
        uid = userIds[row['userId']]
        iid = itemIds[row['movieId']]

        rating_pred = mean + user_bias[uid] + item_bias[iid]
        rating_pred = max(0, min(5, rating_pred))

        if hasattr(rating_pred, 'item'):
            rating_pred = rating_pred.item()

        writer.writerow({
            'id': f"{row['userId']}_{row['movieId']}",
            'rating': rating_pred
        })

In [ ]:
!head bias_solution.csv

id,rating
64777_12355,3.9157955646514893
64777_1864,4.008310794830322
64777_7104,4.043219089508057
64777_22572,4.1636152267456055
64777_6751,3.5647976398468018
42001_994,3.8256642818450928
42001_11270,3.824188470840454
42001_23855,2.917877674102783
42001_19015,3.494414806365967


## Bias 모델 - Gradient Descent (Pytorch)

In [ ]:
import torch

lr = 0.013
lmd = 0.0001

users = torch.LongTensor(users)
items = torch.LongTensor(items)
ratings = torch.FloatTensor(ratings)

user_bias = torch.zeros(n_users, requires_grad=True)
item_bias = torch.zeros(n_items, requires_grad=True)

optim = torch.optim.Adam(params=[user_bias, item_bias], lr=lr)
mse = torch.nn.MSELoss()

for epoch in range(1000):
  h = mean + user_bias[users] + item_bias[items]
  cost = mse(h, ratings)
  cost += lmd * (user_bias**2).sum()
  cost += lmd * (item_bias**2).sum()

  optim.zero_grad()
  cost.backward()
  optim.step()

  print(f"epoch:{epoch}, mse:{cost.item()}")

epoch:0, mse:1.104691743850708
epoch:1, mse:1.0862245559692383
epoch:2, mse:1.0690730810165405
epoch:3, mse:1.0531818866729736
epoch:4, mse:1.0384575128555298
epoch:5, mse:1.0248113870620728
epoch:6, mse:1.012168526649475
epoch:7, mse:1.0004591941833496
epoch:8, mse:0.9896136522293091
epoch:9, mse:0.979563295841217
epoch:10, mse:0.9702459573745728
epoch:11, mse:0.9616056680679321
epoch:12, mse:0.9535900950431824
epoch:13, mse:0.9461482763290405
epoch:14, mse:0.9392326474189758
epoch:15, mse:0.9327995181083679
epoch:16, mse:0.9268097877502441
epoch:17, mse:0.9212280511856079
epoch:18, mse:0.9160216450691223
epoch:19, mse:0.9111603498458862
epoch:20, mse:0.9066172242164612
epoch:21, mse:0.9023684859275818
epoch:22, mse:0.8983917832374573
epoch:23, mse:0.8946675658226013
epoch:24, mse:0.8911774158477783
epoch:25, mse:0.8879050016403198
epoch:26, mse:0.8848351240158081
epoch:27, mse:0.8819544315338135
epoch:28, mse:0.8792505264282227
epoch:29, mse:0.8767119646072388
epoch:30, mse:0.8743281

##Bias 모델 - Alterating Least Squares

In [ ]:
lmd = 10

mean = ratings.mean()
user_bias = np.zeros(n_users)
item_bias = np.zeros(n_items)

for epoch in range(50):
  h = mean + user_bias[users] + item_bias[items]
  mse = ((h-ratings)**2).mean()
  print(f"epoch:{epoch}, mse:{mse}")

  tmp = rating - (mean+item_bias[items])
  user_bias = np.bincount(users, weights=tmp) / (np.bincount(users)+lmd)
  tmp = ratings - (mean+user_bias[users])
  item_bias = np.bincount(items,weights=tmp) / (np.bincount(items)+lmd)

/tmp/ipykernel_651/3900174083.py:8: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  h = mean + user_bias[users] + item_bias[items]
/tmp/ipykernel_651/3900174083.py:12: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  tmp = rating - (mean+item_bias[items])
/tmp/ipykernel_651/3900174083.py:14: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  tmp = ratings - (mean+user_bias[users])


epoch:0, mse:1.1046917649015608
epoch:1, mse:0.8996394913505591
epoch:2, mse:0.9220230585082416
epoch:3, mse:0.9285462313872865
epoch:4, mse:0.9294861800593832
epoch:5, mse:0.9293381960795087
epoch:6, mse:0.9290280600705545
epoch:7, mse:0.9287374333521019
epoch:8, mse:0.928497329198985
epoch:9, mse:0.9283064329728646
epoch:10, mse:0.9281568103868343
epoch:11, mse:0.9280401406294791
epoch:12, mse:0.9279492504134081
epoch:13, mse:0.9278783508121231
epoch:14, mse:0.9278229011351741
epoch:15, mse:0.9277793868046812
epoch:16, mse:0.9277451039467701
epoch:17, mse:0.9277179771779674
epoch:18, mse:0.9276964138514691
epoch:19, mse:0.9276791905108335
epoch:20, mse:0.9276653655933494
epoch:21, mse:0.9276542127547523
epoch:22, mse:0.9276451701007671
epoch:23, mse:0.9276378015750799
epoch:24, mse:0.9276317675940536
epoch:25, mse:0.9276268026994334
epoch:26, mse:0.9276226985331094
epoch:27, mse:0.9276192908477738
epoch:28, mse:0.927616449579621
epoch:29, mse:0.9276140712461387
epoch:30, mse:0.927612